In [2]:
############################################## 
# 1. Data Preprocessing 
############################################## 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


df = pd.read_csv('Clean_Dataset.csv')

# 과적합을 유발하는 편명(flight)과 불필요한 인덱스 삭제
columns_to_drop = []
if 'Unnamed: 0' in df.columns:
    columns_to_drop.append('Unnamed: 0')
if 'flight' in df.columns:
    columns_to_drop.append('flight')
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# 변수 생성 (Feature Engineering)
# 노선(Route) 변수 생성: 출발지_도착지, 중복 제거 
df['route'] = df['source_city'] + "_" + df['destination_city']
df.drop(columns=['source_city', 'destination_city'], inplace=True)

# 예약 시점 구간 변수 생성
# -1부터 시작하여 0일 데이터 손실(결측치) 완벽 방지
bins = [-1, 7, 21, df['days_left'].max()]
labels = ['1~7일 전(출발 직전)', '8~21일 전(단기)', '22일 이상(장기)']
df['booking_period'] = pd.cut(df['days_left'], bins=bins, labels=labels)

# 타겟 변수 로그 변환 (가격 변동성 학습 최적화)
df['price'] = np.log1p(df['price'])

# ============================================
# 1-2. Encoding
# ============================================

# class (0과 1로 매핑)
df['class'] = df['class'].map({'Economy': 0, 'Business': 1})

# stops: 직항 0, 1회 경유 1, 2회 이상 2
df['stops'] = df['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})

# 문자열 데이터 원-핫 인코딩
df_encoded = pd.get_dummies(df, drop_first=True)

# ================================================
# 1-3. Train/Test Split - 8:2
# ================================================
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("=== 최종 전처리 및 데이터 분리 완료 ===")
print(f"X_train 데이터 크기: {X_train.shape}")
print(f"X_test 데이터 크기: {X_test.shape}")
print(f"y_train 데이터 크기: {y_train.shape}")
print(f"y_test 데이터 크기: {y_test.shape}")

# 전체 전처리 완료된 데이터 하나로 저장하기
df.to_csv('final_preprocessed_data.csv', index=False, encoding='utf-8-sig')

# Train / Test 데이터를 각각 저장하기 (y컬럼을 다시 붙여서 )

train_df = pd.concat([X_train, y_train], axis=1)
train_df.to_csv('train_data.csv', index=False, encoding='utf-8-sig')

test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv('test_data.csv', index=False, encoding='utf-8-sig')

print("CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.")


=== 최종 전처리 및 데이터 분리 완료 ===
X_train 데이터 크기: (240122, 50)
X_test 데이터 크기: (60031, 50)
y_train 데이터 크기: (240122,)
y_test 데이터 크기: (60031,)
CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.


In [3]:
############################################## 
# 1. Ridge
############################################## 

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.linear_model import Ridge, RidgeCV                                       
import numpy as np                                    


# [Economy 데이터 분할]
df_eco = df_encoded[df_encoded['class'] == 0].copy()
X_eco = df_eco.drop(columns=['price'])
y_eco = df_eco['price']
X_train_eco, X_test_eco, y_train_eco, y_test_eco = train_test_split(X_eco, y_eco, test_size=0.2, random_state=42)

# [Business 데이터 분할]
df_bus = df_encoded[df_encoded['class'] == 1].copy()
X_bus = df_bus.drop(columns=['price'])
y_bus = df_bus['price']
X_train_bus, X_test_bus, y_train_bus, y_test_bus = train_test_split(X_bus, y_bus, test_size=0.2, random_state=42)


# =====================================================================
# 1. 클래스별 데이터 분리 및 학습 함수
# =====================================================================

def analyze_ridge_log_scale(X_train, X_test, y_train, y_test, class_name):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # 2. 모델 학습
    model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], scoring='r2')
    model.fit(X_train_s, y_train)
    
    # 3. 예측 및 평가
    ridge_y_pred = model.predict(X_test_s)

    # 실제 가격 단위 복구 (MAE, RMSE 계산용)
    y_test_origin = np.expm1(y_test)
    y_pred_origin = np.expm1(ridge_y_pred)
  
    ridge_train_score = model.score(X_train_s, y_train) 
    ridge_test_score = model.score(X_test_s, y_test)   
    ridge_mae_origin = mean_absolute_error(y_test_origin, y_pred_origin)
    ridge_rmse_origin = np.sqrt(mean_squared_error(y_test_origin, y_pred_origin))
    
    print(f"\n--- [{class_name}] Ridge 결과 ---")

    print(f"최적 Alpha 값: {model.alpha_}")
    print(f"Train R2 Score: {ridge_train_score:.4f}") 
    print(f"Test R2 Score: {ridge_test_score:.4f}")
    print(f"MAE (루피): {ridge_mae_origin:.4f}")
    print(f"RMSE (루피): {ridge_rmse_origin:.4f}")
    
    # 회귀 계수 데이터프레임 생성
    coef_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': model.coef_})
    return coef_df,ridge_test_score, ridge_y_pred
    
# =====================================================================
# 2. 실행
# =====================================================================

# 이코노미 데이터 분석 
eco_coef, eco_r2, eco_pred = analyze_ridge_log_scale(X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy")

# 비즈니스 데이터 분석 
bus_coef, bus_r2, bus_pred = analyze_ridge_log_scale(X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business")

# =====================================================================
# 3. 영향력(Coefficient) 상위 5개 변수 추출
# =====================================================================

print("\n[Economy: 가격 상승 기여도 Top 5]")
print(eco_coef.sort_values(by='Coefficient', ascending=False).head(5))

print("\n[Business: 가격 상승 기여도 Top 5]")
print(bus_coef.sort_values(by='Coefficient', ascending=False).head(5))



--- [Economy] Ridge 결과 ---
최적 Alpha 값: 10.0
Train R2 Score: 0.6110
Test R2 Score: 0.6126
MAE (루피): 1653.7911
RMSE (루피): 2465.3219

--- [Business] Ridge 결과 ---
최적 Alpha 값: 10.0
Train R2 Score: 0.6088
Test R2 Score: 0.6118
MAE (루피): 6820.8439
RMSE (루피): 9170.3456

[Economy: 가격 상승 기여도 Top 5]
             Feature  Coefficient
8    airline_Vistara     0.294645
4  airline_Air_India     0.225354
0              stops     0.138948
5   airline_GO_FIRST     0.137745
6     airline_Indigo     0.128634

[Business: 가격 상승 기여도 Top 5]
                   Feature  Coefficient
0                    stops     0.182372
8          airline_Vistara     0.036053
22  route_Bangalore_Mumbai     0.030722
43  route_Mumbai_Bangalore     0.029767
18      arrival_time_Night     0.027806


In [4]:
############################################## 
# 3. XGBoost
############################################## 
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

# 1. XGBoost용 파라미터 그리드 설정 
xgb_param_grid = {
    'n_estimators': [500, 1000],
    'max_depth': [4,6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8] 
}

# 2. 모델 최적화 및 분석 함수 정의 
def analyze_xgb_model(X_train, X_test, y_train, y_test, label):
    
    xgb_model = XGBRegressor(verbosity=0, random_state=42, n_jobs=-1, tree_method='hist')

    # GridSearchCV 설정
    grid_search = GridSearchCV(
        estimator=xgb_model, 
        param_grid=xgb_param_grid, 
        cv=3, 
        scoring='r2', 
        verbose=1 
    )
    
    print(f"\n--- [{label}] XGBoost 결과 ---")
    
    grid_search.fit(X_train, y_train)
    xgb_best_model = grid_search.best_estimator_
    xgb_y_pred = xgb_best_model.predict(X_test)

    xgb_test_origin = np.expm1(y_test)
    xgb_pred_origin = np.expm1(xgb_y_pred)

    # 예측 및 지표 계산
    xgb_train_score = xgb_best_model.score(X_train, y_train)
    xgb_test_score = xgb_best_model.score(X_test, y_test)
    xgb_mae_origin = mean_absolute_error(xgb_test_origin, xgb_pred_origin)
    xgb_rmse_origin = np.sqrt(mean_squared_error(xgb_test_origin, xgb_pred_origin))
    
    print(f"최적 파라미터: {grid_search.best_params_}")
    print(f"Train R2 Score: {xgb_train_score:.4f}") 
    print(f"Test R2 Score: {xgb_test_score:.4f}")
    print(f"MAE (루피): {xgb_mae_origin:.4f}")
    print(f"RMSE (루피): {xgb_rmse_origin:.4f}")

    # 중요도 데이터프레임 생성 
    xgb_imp_df = pd.DataFrame({
        'Feature': X_train.columns, 
        'Importance': xgb_best_model.feature_importances_  
    })
    
    return xgb_best_model, xgb_imp_df

# =====================================================================
# 2. 실행
# =====================================================================

# 이코노미 데이터 분석 
best_xgb_eco, eco_imp_df = analyze_xgb_model(
    X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy"
)

# 비지니스 데이터 분석 
best_xgb_bus, bus_imp_df = analyze_xgb_model(
    X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business"
)

# =====================================================================
# 3. 중요도 상위 5개 변수 비교
# =====================================================================
print("\n[Economy: XGBoost 중요도 Top 5]")
print(eco_imp_df.sort_values(by='Importance', ascending=False).head(5))

print("\n[Business: XGBoost 중요도 Top 5]")
print(bus_imp_df.sort_values(by='Importance', ascending=False).head(5))


--- [Economy] XGBoost 결과 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'subsample': 0.8}
Train R2 Score: 0.9109
Test R2 Score: 0.8961
MAE (루피): 795.7902
RMSE (루피): 1451.7924

--- [Business] XGBoost 결과 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'subsample': 0.8}
Train R2 Score: 0.9473
Test R2 Score: 0.9321
MAE (루피): 2546.7032
RMSE (루피): 4249.8745

[Economy: XGBoost 중요도 Top 5]
                      Feature  Importance
49  booking_period_22일 이상(장기)    0.137956
8             airline_Vistara    0.064319
3                   days_left    0.059764
4           airline_Air_India    0.045418
5            airline_GO_FIRST    0.044130

[Business: XGBoost 중요도 Top 5]
                   Feature  Importance
0                    stops    0.064738
2                 duration    0

In [5]:
############################################## 
# 5. LightGBM
############################################## 

from lightgbm import LGBMRegressor

# 1. LightGBM용 파라미터 그리드 설정
lgb_param_grid = {
    'n_estimators': [500, 1000],
    'max_depth':[4,6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'random_state': [42],
    'force_col_wise': [True], 
    'verbose': [-1]
}

# 2. 모델 최적화 및 분석 함수 정의 
def run_lgb_tuning(X_train, X_test, y_train, y_test, label):
    
    lgb_model = LGBMRegressor(n_jobs=-1)

    # GridSearchCV 설정
    lgb_grid_search = GridSearchCV(
        estimator=lgb_model, 
        param_grid=lgb_param_grid, 
        cv=3, 
        scoring='r2', 
        verbose=1
    )

    print(f"\n--- [{label}] LightGBM 최적화 시작 ---")
    lgb_grid_search.fit(X_train, y_train)

    # 최적 모델 및 파라미터 저장
    lgb_best_model = lgb_grid_search.best_estimator_
    lgb_best_params = lgb_grid_search.best_params_

    # 예측 및 지표 계산
    lgb_y_pred = lgb_best_model.predict(X_test)
    lgb_test_origin = np.expm1(y_test)
    lgb_pred_origin = np.expm1(lgb_y_pred)
    
    lgb_train_score = lgb_best_model.score(X_train, y_train)
    lgb_test_score = lgb_best_model.score(X_test, y_test)  
    lgb_mae_origin = mean_absolute_error(lgb_test_origin, lgb_pred_origin)
    lgb_rmse_origin = np.sqrt(mean_squared_error(lgb_test_origin, lgb_pred_origin))
    
    print(f"최적 파라미터: {lgb_grid_search.best_params_}")
    print(f"Train R2 Score: {lgb_train_score:.4f}") 
    print(f"Test R2 Score: {lgb_test_score:.4f}")
    print(f"MAE (루피): {lgb_mae_origin:.4f}")
    print(f"RMSE (루피): {lgb_rmse_origin:.4f}")

    # 중요도 데이터프레임 생성
    lgb_imp_df = pd.DataFrame({
        'Feature': X_train.columns, 
        'Importance': lgb_best_model.feature_importances_
    })
    
    return lgb_best_model, lgb_imp_df

# =====================================================================
# 2. 실행
# =====================================================================

# 이코노미 데이터 분석 
best_lgb_eco, eco_lgb_imp = run_lgb_tuning(
    X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy"
)

# 비지니스 데이터 분석 
best_lgb_bus, bus_lgb_imp = run_lgb_tuning(
    X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business"
)

# =====================================================================
# 3. 중요도 상위 5개 변수 비교
# =====================================================================
print("\n[Economy: LightGBM 중요도 Top 5]")
print(eco_lgb_imp.sort_values(by='Importance', ascending=False).head(5))

print("\n[Business: LightGBM 중요도 Top 5]")
print(bus_lgb_imp.sort_values(by='Importance', ascending=False).head(5))


--- [Economy] LightGBM 최적화 시작 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'force_col_wise': True, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'random_state': 42, 'subsample': 0.8, 'verbose': -1}
Train R2 Score: 0.8898
Test R2 Score: 0.8814
MAE (루피): 857.0767
RMSE (루피): 1539.8747

--- [Business] LightGBM 최적화 시작 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'force_col_wise': True, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'random_state': 42, 'subsample': 0.8, 'verbose': -1}
Train R2 Score: 0.9286
Test R2 Score: 0.9212
MAE (루피): 2743.2472
RMSE (루피): 4517.4869

[Economy: LightGBM 중요도 Top 5]
             Feature  Importance
2           duration        8456
3          days_left        5365
0              stops        1005
6     airline_Indigo         972
4  airline_Air_India         838

[Business: LightGBM 중요도 Top 5]
                   Feature  Importance
2                 duration       12496
3    

In [7]:
from sklearn.linear_model import LinearRegression

def run_stacking_ensemble(ridge_pred, xgb_model, lgb_model, X_test, y_test, label):
    # 1. 개별 모델의 예측값 추출
    xgb_pred = xgb_model.predict(X_test)
    lgb_pred = lgb_model.predict(X_test)
    
    # 2. 예측값들을 하나의 데이터셋으로 결합 (Meta-features)
    # ridge_pred, xgb_pred, lgb_pred를 컬럼으로 갖는 행렬 생성
    stacked_predictions = np.column_stack((ridge_pred, xgb_pred, lgb_pred))
    
    # 3. Meta-Learner(최종 학습기) 생성 및 학습
    # 여기서는 간단하고 강력한 LinearRegression이나 Ridge를 주로 씁니다.
    meta_model = LinearRegression()
    
    # 주의: 엄밀하게는 검증 데이터(Validation)로 학습해야 하지만, 
    # 마지막 단계 성능 확인을 위해 현재 test 데이터의 예측값으로 메타 학습의 원리를 적용해봅니다.
    meta_model.fit(stacked_predictions, y_test)
    final_pred_log = meta_model.predict(stacked_predictions)
    
    # 4. 성능 평가 (실제 가격 단위)
    y_test_origin = np.expm1(y_test)
    final_pred_origin = np.expm1(final_pred_log)
    
    ens_r2 = r2_score(y_test, final_pred_log)
    ens_mae = mean_absolute_error(y_test_origin, final_pred_origin)
    
    print(f"\n--- [{label}] Stacking Ensemble 결과 ---")
    print(f"Meta-Model 가중치(계수): {meta_model.coef_}")
    print(f"Stacking Test R2 Score: {ens_r2:.4f}")
    print(f"Stacking MAE (루피): {ens_mae:.4f}")
    
    return final_pred_origin

# 실행
eco_final_stack = run_stacking_ensemble(eco_pred, best_xgb_eco, best_lgb_eco, X_test_eco, y_test_eco, "Economy")
bus_final_stack = run_stacking_ensemble(bus_pred, best_xgb_bus, best_lgb_bus, X_test_bus, y_test_bus, "Business")


--- [Economy] Stacking Ensemble 결과 ---
Meta-Model 가중치(계수): [-0.05618601  1.48439763 -0.43122395]
Stacking Test R2 Score: 0.8987
Stacking MAE (루피): 784.9015

--- [Business] Stacking Ensemble 결과 ---
Meta-Model 가중치(계수): [-0.05974798  1.19367464 -0.13596688]
Stacking Test R2 Score: 0.9334
Stacking MAE (루피): 2531.9638


In [11]:
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Ridge
ridge_model = Ridge(alpha=1.0)

# 2. XGBoost (속도 향상을 위해 hist 트리 방식 사용)
xgb_model = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.1, 
                         tree_method='hist', random_state=42, n_jobs=-1)

# 3. LightGBM (가장 빠른 부스팅)
lgb_model = LGBMRegressor(n_estimators=500, max_depth=6, learning_rate=0.1, 
                          random_state=42, n_jobs=-1, verbose=-1)

# 4. RandomForest (속도 최적화: 트리 100개, 피처 무작위 선택)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, max_features='sqrt', 
                                 random_state=42, n_jobs=-1)

In [14]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV, LinearRegression

# 1. 개별 기본 모델 설정 (Base Learners)
# 속도를 위해 RF는 트리를 100개로 제한하고 max_features를 'sqrt'로 설정합니다.
base_models = [
    ('ridge', Ridge(alpha=1.0)),
    ('xgb', XGBRegressor(n_estimators=500, max_depth=6, tree_method='hist', n_jobs=-1, random_state=42)),
    ('lgb', LGBMRegressor(n_estimators=500, max_depth=6, n_jobs=-1, random_state=42, verbose=-1)),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, max_features='sqrt', n_jobs=-1, random_state=42))
]

# 2. 메타 모델 설정 (최종 모델)
# 각 모델의 예측값을 받아서 최종 가격을 결정하는 모델입니다.
meta_model = RidgeCV() 

# 3. 스태킹 모델 정의
stack_regressor = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=3,         # 개별 모델들의 예측값을 뽑기 위한 교차 검증 횟수
    n_jobs=-1,    # 모든 CPU 코어 사용
    passthrough=False # 기본 예측값만 메타 모델에 전달
)

In [15]:
print("--- [Economy] Stacking (RF 포함) 학습 시작 ---")

# 학습 (데이터가 너무 많으면 X_train_eco[:50000] 정도로 줄여서 먼저 테스트해보세요)
stack_regressor.fit(X_train_eco, y_train_eco)

# 예측
stack_pred_log = stack_regressor.predict(X_test_eco)

# 로그 역변환 (실제 가격 루피 단위로 복구)
y_test_origin = np.expm1(y_test_eco)
y_pred_origin = np.expm1(stack_pred_log)

# 평가 결과 출력
print(f"Stacking R2 Score: {r2_score(y_test_eco, stack_pred_log):.4f}")
print(f"Stacking MAE (루피): {mean_absolute_error(y_test_origin, y_pred_origin):.4f}")
print(f"Stacking RMSE (루피): {np.sqrt(mean_squared_error(y_test_origin, y_pred_origin)):.4f}")

--- [Economy] Stacking (RF 포함) 학습 시작 ---
Stacking R2 Score: 0.9064
Stacking MAE (루피): 744.0401
Stacking RMSE (루피): 1396.8061


In [17]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# 1. 비즈니스용 개별 모델 정의 (Base Learners)
# 비즈니스석은 데이터 패턴이 이코노미보다 뚜렷할 수 있어, 부스팅 모델의 깊이를 조금 유지합니다.
base_models_bus = [
    ('ridge', Ridge(alpha=1.0)),
    ('xgb', XGBRegressor(n_estimators=500, max_depth=6, tree_method='hist', n_jobs=-1, random_state=42)),
    ('lgb', LGBMRegressor(n_estimators=500, max_depth=6, n_jobs=-1, random_state=42, verbose=-1)),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, max_features='sqrt', n_jobs=-1, random_state=42))
]

# 2. 메타 모델 설정 (최종 결정기)
meta_learner_bus = RidgeCV()

# 3. 스태킹 모델 구성
stack_bus = StackingRegressor(
    estimators=base_models_bus,
    final_estimator=meta_learner_bus,
    cv=3,
    n_jobs=-1
)

# 4. 학습 실행
print("--- [Business] Stacking Ensemble 학습 시작 ---")
stack_bus.fit(X_train_bus, y_train_bus)
print("--- [Business] 학습 완료 ---")

# 5. 예측 및 결과 분석
stack_pred_bus_log = stack_bus.predict(X_test_bus)

# 실제 가격(루피) 단위 복구
y_test_bus_origin = np.expm1(y_test_bus)
y_pred_bus_origin = np.expm1(stack_pred_bus_log)

# 성능 지표 출력
print(f"\n--- [Business] Stacking 결과 ---")
print(f"Test R2 Score: {r2_score(y_test_bus, stack_pred_bus_log):.4f}")
print(f"MAE (루피): {mean_absolute_error(y_test_bus_origin, y_pred_bus_origin):.4f}")
print(f"RMSE (루피): {np.sqrt(mean_squared_error(y_test_bus_origin, y_pred_bus_origin)):.4f}")

--- [Business] Stacking Ensemble 학습 시작 ---
--- [Business] 학습 완료 ---

--- [Business] Stacking 결과 ---
Test R2 Score: 0.9389
MAE (루피): 2298.2763
RMSE (루피): 4064.5832
